# SignalDesk: inspect the data and measured results

Run this notebook from the project root or `notebooks/` after the setup in README. It audits the saved experiment; it does not tune against the test set. Training lives in `signaldesk.train`.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import plotly.express as px

ROOT = Path.cwd() if (Path.cwd() / 'pyproject.toml').exists() else Path.cwd().parent
metrics = json.loads((ROOT / 'artifacts/metrics.json').read_text())
train = pd.read_csv(ROOT / 'data/raw/train.csv')
test = pd.read_csv(ROOT / 'data/raw/test.csv')
print(f'Original train: {len(train):,}; official test: {len(test):,}; classes: {train.category.nunique()}')
train.head()

## Class balance and query length

BANKING77 is a single-domain, English intent benchmark. Its official test set has 40 examples per class. A real support queue will have a different class distribution and can contain multiple intents, personal information, or unknown issues.

In [ ]:
counts = train.category.value_counts().rename_axis('intent').reset_index(name='queries')
px.bar(counts.head(20), x='queries', y='intent', orientation='h', title='Largest training classes').show()
px.histogram(train.assign(words=train.text.str.split().str.len()), x='words', nbins=50, title='Query length in words').show()

## Leakage audit and split boundaries

Only the fit partition learns vocabulary and classifier coefficients. Calibration, candidate selection, and policy validation each have their own partition. Exact normalized duplicates are removed; this does not detect all paraphrases.

In [ ]:
print(json.dumps(metrics['leakage_audit'], indent=2))
pd.Series(metrics['split_sizes'], name='examples').to_frame()

## Candidate comparison and official test results

Model choice uses selection-partition macro F1. The test results below evaluate only the winner with the fixed review threshold. Inspect the gap between whole-test accuracy and accepted-ticket accuracy, along with coverage.

In [ ]:
pd.json_normalize(metrics['candidates'])

In [ ]:
print('Selected model:', metrics['selected_model'])
print('Whole-test metrics:', json.dumps(metrics['test'], indent=2))
print('Fixed review policy:', json.dumps(metrics['policy'], indent=2))

## Calibration and error analysis

Top-label expected calibration error is a binned summary. It can hide class-specific problems. Confusion counts below are diagnostic; changing the model based on these results would require a new untouched evaluation set.

In [ ]:
reliability = pd.DataFrame(metrics['reliability_bins']).dropna(subset=['mean_confidence'])
fig = px.scatter(reliability, x='mean_confidence', y='accuracy', size='count', range_x=[0,1], range_y=[0,1], title='Top-label reliability on official test')
fig.add_shape(type='line', x0=0, y0=0, x1=1, y1=1, line=dict(dash='dash'))
fig.show()
pd.DataFrame(metrics['confusion_pairs']).head(15)

In [ ]:
pd.DataFrame(metrics['per_class']).sort_values('f1').head(15)

## Inspect a prediction

Neighbors are public fit-partition queries matched by word TF-IDF cosine similarity. Feature contributions explain the raw linear score, not calibrated probability or causality.

In [ ]:
from signaldesk.modeling import load_bundle, predict
bundle = load_bundle(ROOT / 'artifacts')
predict(bundle, ['The ATM kept my card and I cannot get it back.'])[0]